In [0]:
df_silver = spark.read.format("delta").load(
    "abfss://silver@stnyc311.dfs.core.windows.net/nyc311/"
)

In [0]:
df_gold_complaints = (
    df_silver
    .groupBy("complaint_type")
    .count()
    .withColumnRenamed("count", "request_count")
    .orderBy("request_count", ascending=False)
)

In [0]:
df_gold_complaints.show(10, truncate=False)

In [0]:
df_gold_complaints.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@stnyc311.dfs.core.windows.net/complaint_counts/")

In [0]:
df_gold_boroughs = (
    df_silver
    .groupBy("borough")
    .count()
    .withColumnRenamed("count", "request_count")
    .orderBy("request_count", ascending=False)
)

In [0]:
df_gold_boroughs.show(truncate=False)

In [0]:
df_gold_boroughs.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@stnyc311.dfs.core.windows.net/borough_counts/")

In [0]:
df_resolved_valid = df_silver.filter(
    df_silver.resolution_time_flag == "VALID"
)

In [0]:
df_resolved_valid.count()

In [0]:
df_resolved_valid.select("resolution_hours").summary(
    "count", "min", "25%", "50%", "75%", "mean", "max"
).show()

In [0]:
from pyspark.sql import functions as F

df_gold_resolution = (
    df_resolved_valid
    .groupBy("complaint_type")
    .agg(
        F.expr("percentile_approx(resolution_hours, 0.5)")
         .alias("median_resolution_hours"),
        F.count("*").alias("resolved_request_count")
    )
    .orderBy("median_resolution_hours", ascending=False)
)

In [0]:
df_gold_resolution.show(15, truncate=False)

In [0]:
df_gold_resolution.select(
    F.count("*").alias("total_complaint_types"),
    F.sum((F.col("resolved_request_count") >= 20).cast("int")).alias("types_20_plus"),
    F.sum((F.col("resolved_request_count") >= 50).cast("int")).alias("types_50_plus"),
    F.sum((F.col("resolved_request_count") >= 100).cast("int")).alias("types_100_plus")
).show()

In [0]:
df_gold_resolution_filtered = (
    df_gold_resolution
    .filter(F.col("resolved_request_count") >= 50)
    .orderBy("median_resolution_hours", ascending=False)
)

In [0]:
df_gold_resolution_filtered = (
    df_gold_resolution
    .filter(F.col("resolved_request_count") >= 50)
    .orderBy("median_resolution_hours", ascending=False)
)

In [0]:
df_gold_resolution_filtered.show(15, truncate=False)

In [0]:
df_gold_resolution_filtered.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@stnyc311.dfs.core.windows.net/resolution_by_complaint/")